# 09 — RAG Ingestion Pipeline
Explore document loaders, chunker, embedder, and pgvector store.

In [ ]:
import sys; sys.path.insert(0, '/home/claude/codebase/code/src', 0); sys.path.insert(0, '/home/claude/codebase/code')
import os; os.environ['ENABLE_MOCK']='true'

## Document Loaders

In [ ]:
from ingestion.loaders import load_document
import tempfile, os

# Create a temp text file to demo loading
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write('Gross Retention Rate (GRR) measures the percentage of recurring revenue retained from existing customers over a period, excluding expansion revenue.\n\nGRR = (MRR at end - expansion) / MRR at start\n\nA healthy SaaS GRR is typically above 85%.')
    tmp_path = f.name

docs = load_document(tmp_path)
print(f'Documents loaded: {len(docs)}')
print(f'Content preview: {docs[0].page_content[:120]}')
print(f'Metadata: {docs[0].metadata}')
os.unlink(tmp_path)

### Supported file types

In [ ]:
print('Supported extensions and their loaders:')
print('  .pdf  → PyPDFLoader (langchain-community)')
print('  .docx → Docx2txtLoader (langchain-community)')
print('  .pptx → UnstructuredPowerPointLoader')
print('  .xlsx → UnstructuredExcelLoader')
print('  .txt  → TextLoader (fallback)')
print()
print('Load raises ValueError for unsupported extensions.')
try:
    load_document('file.xyz')
except ValueError as e:
    print(f'Unsupported: {e}')

## Chunker

In [ ]:
from ingestion.chunker import chunk_documents
from langchain_core.documents import Document

# Create sample documents
sample_docs = [
    Document(
        page_content='Gross Retention Rate (GRR) is a key SaaS metric. '
            'It measures revenue retained from existing customers, excluding expansion. '
            'A healthy GRR is above 85%. Low GRR indicates churn risk. '
            'Monitor GRR monthly alongside NRR for full picture. '
            'GRR is calculated as: (Start MRR - Churn MRR) / Start MRR.',
        metadata={'source': 'governance-policy.pdf'}
    ),
]

chunks = chunk_documents(sample_docs, source='governance-policy.pdf', product='retention')
print(f'Input docs: {len(sample_docs)}')
print(f'Output chunks: {len(chunks)}')
for i, c in enumerate(chunks):
    print(f'\nChunk {i+1}:')
    print(f'  Text: {c.page_content[:80]}...')
    print(f'  Metadata: {c.metadata}')

## Embedder (requires OPENAI_API_KEY — shows interface only)

In [ ]:
# Interface demonstration — requires real OPENAI_API_KEY to actually embed
from ingestion.embedder import get_embedder

api_key = os.getenv('OPENAI_API_KEY', '')
if api_key:
    embedder = get_embedder()
    vectors = embedder.embed_documents(['What is GRR?', 'What is CAC?'])
    print(f'Embeddings shape: {len(vectors)} x {len(vectors[0])}')
else:
    print('OPENAI_API_KEY not set — embedder would produce 1536-dim vectors')
    print('Model: text-embedding-3-small (batch size: 100)')
    print('Usage: embedder.embed_documents(["text1", "text2"])')

## Airflow DAGs overview

In [ ]:
print('Airflow DAGs in /dags/:')
dags = [
    ('file_ingestion_dag.py',    'FileSensor on docs/ → auto-ingest on file drop'),
    ('on_demand_ingest_dag.py',  'API-triggered via POST /ingest (conf.filepath)'),
    ('collibra_sync_dag.py',     'Daily 06:00 UTC — pull Collibra assets → embed'),
    ('nightly_refresh_dag.py',   'Nightly 02:00 UTC — hash-diff → re-embed changed chunks'),
]
for name, desc in dags:
    print(f'  📋 {name}')
    print(f'     {desc}')

## End-to-end ingestion flow

In [ ]:
print('Full ingestion pipeline:')
steps = [
    ('1. Load', 'loaders.load_document(filepath) → List[Document]'),
    ('2. Chunk', 'chunker.chunk_documents(docs, source, product) → List[Document]'),
    ('3. Embed', 'embedder.embed_documents([chunk.page_content, ...]) → List[List[float]]'),
    ('4. Store', 'store.upsert_chunks(chunks, vectors) → dedup via SHA-256 + pgvector'),
]
for step, desc in steps:
    print(f'  {step}: {desc}')